In [1]:
import torch
import numpy as np
from torch.utils.data import Dataset

from utils.gpu_check import get_device

In [2]:
device, CUDA = get_device()

Gen RAM Free: 59.7 GB | Proc size: 572.9 MB
GPU RAM Free: 14568MB | Used: 1241MB | Util   8% | Total 16303MB
Using device: cuda


In [3]:
from structures.medical_sequence import MedicalSequence

In [4]:
def print_head_of_sequences(sequences: list[MedicalSequence]):
    print_size = min(len(sequences), 5)
    for seq in sequences[:print_size]:
        print(seq)

In [5]:
from dataset.sequence_generator import SequenceGenerator
from dataset.medical_sequence_dataset import MedicalSequenceDataset


In [ ]:
dataset = MedicalSequenceDataset.from_generator(
    generator=SequenceGenerator(),
    n_sequences=10000,
    max_steps=8,
    pad_cabinet_token_id=0,
    pad_to_max_length=True,
    encode_condition_pad_for_model=True,
)

In [7]:
print(len(dataset))
print(dataset.summary())

10000
{'num_sequences': 10000, 'condition_feature_sizes_raw': [3, 2, 4], 'condition_feature_sizes_for_model': [4, 3, 5], 'max_conditions_per_sequence': 9, 'max_cabinets_per_sequence': 8, 'pad_cabinet_token_id': 0, 'encode_condition_pad_for_model': True}


In [8]:
from torch.utils.data import DataLoader, random_split
import torch.nn as nn

from medical_transformer.medical_transformer_wrapper import TransformerWrapper
from encoders.medical_sequence_encoder import MedicalTokenEncoders
from decoders.medical_output_decoder import MedicalOutputHeads

In [9]:
def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device)
        else:
            out[k] = v
    return out

def train_one_epoch(
    model: TransformerWrapper,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> dict[str, float]:
    model.train()

    total_loss = 0.0
    total_cabinet_loss = 0.0
    total_condition_loss = 0.0
    n_batches = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()

        losses = model.compute_losses(batch)
        loss = losses["loss"]

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_cabinet_loss += losses["cabinet_loss"].item()
        total_condition_loss += losses["condition_loss"].item()
        n_batches += 1

    return {
        "loss": total_loss / max(n_batches, 1),
        "cabinet_loss": total_cabinet_loss / max(n_batches, 1),
        "condition_loss": total_condition_loss / max(n_batches, 1),
    }

@torch.no_grad()
def evaluate(
    model: TransformerWrapper,
    loader: DataLoader,
    device: torch.device,
) -> dict[str, float]:
    model.eval()

    total_loss = 0.0
    total_cabinet_loss = 0.0
    total_condition_loss = 0.0
    n_batches = 0

    cabinet_correct = 0
    cabinet_total = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        losses = model.compute_losses(batch)

        total_loss += losses["loss"].item()
        total_cabinet_loss += losses["cabinet_loss"].item()
        total_condition_loss += losses["condition_loss"].item()
        n_batches += 1

        cabinet_logits = losses["cabinet_logits"]      # [B, Tk, V]
        cabinet_targets = batch["cabinets"]            # [B, Tk]
        cabinet_mask = batch["cabinet_mask"].bool()    # [B, Tk]

        preds = cabinet_logits.argmax(dim=-1)
        cabinet_correct += (preds[cabinet_mask] == cabinet_targets[cabinet_mask]).sum().item()
        cabinet_total += cabinet_mask.sum().item()

    return {
        "loss": total_loss / max(n_batches, 1),
        "cabinet_loss": total_cabinet_loss / max(n_batches, 1),
        "condition_loss": total_condition_loss / max(n_batches, 1),
        "cabinet_acc": cabinet_correct / max(cabinet_total, 1),
    }

In [10]:
def infer_num_cabinets(dataset: MedicalSequenceDataset) -> int:
    max_token_id = 0

    for record in dataset._records:
        for cabinet in record.sequence.cabinet_sequence:
            max_token_id = max(max_token_id, cabinet.token_id)

    return max_token_id + 1


train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

feature_cardinalities = dataset.get_condition_feature_vocab_sizes()
num_cabinets = infer_num_cabinets(dataset)
max_seq_len = dataset.max_conditions_len + dataset.max_cabinets_len

d_model = 128
nhead = 8
num_layers = 4
ff_dim = 4 * d_model
dropout = 0.1

token_encoders = MedicalTokenEncoders(
    feature_cardinalities=feature_cardinalities,
    num_cabinets=num_cabinets,
    d_model=d_model,
)

output_heads = MedicalOutputHeads(
    d_model=d_model,
    feature_cardinalities=feature_cardinalities,
    num_cabinets=num_cabinets,
)

encoder_layer = nn.TransformerEncoderLayer(
    d_model=d_model,
    nhead=nhead,
    dim_feedforward=ff_dim,
    dropout=dropout,
    batch_first=True,
    activation="relu",
)

transformer = nn.TransformerEncoder(
    encoder_layer=encoder_layer,
    num_layers=num_layers,
)

model = TransformerWrapper(
    token_encoders=token_encoders,
    transformer=transformer,
    output_heads=output_heads,
    d_model=d_model,
    max_seq_len=max_seq_len,
    use_causal_mask=True,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_cab={train_metrics['cabinet_loss']:.4f} | "
        f"train_cond={train_metrics['condition_loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_cab_acc={val_metrics['cabinet_acc']:.4f}"
    )

batch = next(iter(val_loader))
batch = move_batch_to_device(batch, device)

model.eval()
with torch.no_grad():
    outputs = model(
        conditions=batch["conditions"],
        cabinets=batch["cabinets"],
        padding_mask=batch["padding_mask"],
    )

cabinet_logits = outputs["cabinet_logits"]
cabinet_preds = cabinet_logits.argmax(dim=-1)

print("cabinet_logits.shape:", tuple(cabinet_logits.shape))
print("cabinet_preds.shape :", tuple(cabinet_preds.shape))

Epoch 01 | train_loss=1.4671 | train_cab=0.5141 | train_cond=0.9530 | val_loss=1.3881 | val_cab_acc=0.7661
Epoch 02 | train_loss=1.3762 | train_cab=0.4802 | train_cond=0.8960 | val_loss=1.3693 | val_cab_acc=0.7836
Epoch 03 | train_loss=1.3670 | train_cab=0.4781 | train_cond=0.8889 | val_loss=1.3617 | val_cab_acc=0.7756
Epoch 04 | train_loss=1.3633 | train_cab=0.4763 | train_cond=0.8871 | val_loss=1.3583 | val_cab_acc=0.7788
Epoch 05 | train_loss=1.3586 | train_cab=0.4737 | train_cond=0.8850 | val_loss=1.3579 | val_cab_acc=0.7848
Epoch 06 | train_loss=1.3575 | train_cab=0.4735 | train_cond=0.8840 | val_loss=1.3542 | val_cab_acc=0.7830
Epoch 07 | train_loss=1.3552 | train_cab=0.4723 | train_cond=0.8828 | val_loss=1.3683 | val_cab_acc=0.7763
Epoch 08 | train_loss=1.3551 | train_cab=0.4723 | train_cond=0.8828 | val_loss=1.3518 | val_cab_acc=0.7859
Epoch 09 | train_loss=1.3530 | train_cab=0.4711 | train_cond=0.8819 | val_loss=1.3528 | val_cab_acc=0.7814
Epoch 10 | train_loss=1.3531 | train_